# CHEN — Phase 2: KV-Cache Passing

**The magic step.** Instead of passing *text* between experts, we pass the *KV-cache* — the mathematical memory of the prompt. This is supposed to preserve more nuance than re-encoding text, because the cache is the model's own internal representation, not a lossy English translation.

**Goal of Phase 2:** Measure whether passing KV-caches between heterogeneous small models preserves more nuance than re-encoding text.

We run the same 3-expert sequence as Phase 1, but each expert receives the previous expert's KV-cache (via the backend's `transfer_cache` operation). We compare the latent nuance score against the Phase 1 text-only baseline.

## 1. Setup

In [ ]:
from chen.backends.mock import MockBackend
from chen.core.expert import Expert, ExpertRole
from chen.phases.phase1_cascade import CascadePipeline
from chen.phases.phase2_kv_pass import KVPassPipeline

## 2. Build the same 3-expert garage as Phase 1

In [ ]:
experts = [
    Expert(name="analyst",     role=ExpertRole.ANALYST,     backend=MockBackend(params_m=3_000, role_hint="analyst")),
    Expert(name="reasoner",    role=ExpertRole.REASONER,    backend=MockBackend(params_m=8_000, role_hint="reasoner")),
    Expert(name="synthesizer", role=ExpertRole.SYNTHESIZER, backend=MockBackend(params_m=3_000, role_hint="synthesizer")),
]
sequence = ["analyst", "reasoner", "synthesizer"]

## 3. Run Phase 1 (text handoff) as a baseline

We disable memory augmentation so we isolate the KV-cache variable.

In [ ]:
cascade = CascadePipeline(
    experts=experts,
    sequence=sequence,
    memory_retrieve_k=0,
    write_intermediate_to_memory=False,
)
prompt = "Explain quantum entanglement to a 12-year-old, then derive Bell's inequality step by step."
cascade_result = cascade.run(prompt)
print("=== Phase 1 (text handoff) ===")
print(cascade_result.output[:300])
print()
print(cascade_result.metrics.summary())

## 4. Run Phase 2 (KV-cache handoff)

Same sequence, same prompt — but each expert receives the previous expert's KV-cache instead of text.

In [ ]:
kv_pipe = KVPassPipeline(
    experts=experts,
    sequence=sequence,
    memory_retrieve_k=0,
    write_intermediate_to_memory=False,
)
kv_result = kv_pipe.run(prompt)
print("=== Phase 2 (KV-cache handoff) ===")
print(kv_result.output[:300])
print()
print(kv_result.metrics.summary())

## 5. Compare nuance retention

The `latent_nuance_score` is the fraction of KV transfers that succeeded. A score of 1.0 means every transfer worked; < 1.0 means some experts had to fall back to text.

In a real experiment, you'd also compute a KL-divergence-based probe (see `ARCHITECTURE.md` §5) to measure *how much* nuance was preserved. The MockBackend doesn't have meaningful latents, so the score here just measures transfer success — but the harness is in place for real backends.

In [ ]:
cascade_nuance = cascade_result.metrics.latent_nuance_score
kv_nuance = kv_result.metrics.latent_nuance_score
ratio = kv_nuance / cascade_nuance if cascade_nuance > 0 else float("inf")

print(f"Cascade nuance: {cascade_nuance:.3f}")
print(f"KV-pass nuance: {kv_nuance:.3f}")
print(f"KV-pass retained {ratio:.2f}x the nuance of text-only.")
print()
print(f"KV transfers: {kv_result.metrics.kv_cache_transfers}")
print(f"KV failures:  {kv_result.metrics.kv_cache_failures}")

## 6. Per-expert KV transfer details

In [ ]:
for i, m in enumerate(kv_result.per_expert, 1):
    print(f"Expert {i}: {m.expert_name:14s}")
    print(f"  used KV:              {m.used_kv_cache}")
    print(f"  transfer succeeded:   {m.cache_transfer_succeeded}")
    print(f"  transfer latency:     {m.cache_transfer_ms:.2f} ms")
    print(f"  total latency:        {m.latency_ms:.2f} ms")
    print()

## 7. What does the KV-cache look like?

Let's peek at the actual cache produced by the analyst, before it's transferred to the reasoner.

In [ ]:
analyst = experts[0]
cache = analyst.backend.encode(prompt)
print(cache.summary())
print()
print(f"Layer 0 keys shape: {cache.keys[0].shape}")
print(f"Layer 0 keys dtype: {cache.keys[0].dtype}")
print(f"Layer 0 keys sample (first 3 elements of layer 0, head 0):")
print(cache.keys[0][0, 0, :3])

## 8. What did we just prove?

- The KV-cache produced by Expert 1 was successfully transferred to Expert 2, and from Expert 2 to Expert 3.
- No text was passed between experts — only the latent memory.
- The transfer protocol handles shape mismatches (different layer counts / head dims) by re-encoding.
- The pipeline gracefully falls back to text if a transfer fails.

## 9. What's the catch?

The MockBackend's KV-cache is arbitrary — it doesn't carry real nuance. To prove Phase 2 actually preserves more nuance than text, you need to:

1. Swap `MockBackend` for `HuggingFaceBackend` (install with `pip install -e '.[hf]'`).
2. Use same-family models (e.g. Llama-3-3B → Llama-3-8B) for no-op transfer, or train a learned projection for cross-family.
3. Run a benchmark task with a deterministic grader and compare accuracy between Phase 1 and Phase 2.

The harness is in place — see `examples/run_benchmarks.py`.

## 10. What's next?

**Phase 3** adds **dynamic routing** — a tiny classifier decides which experts to wake up per prompt. Open `03_phase3_routing.ipynb` to see it.